# BetaEarth: Generate Embeddings for Any Area

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/asterisk-labs/beta-earth/blob/main/examples/generate_demo.ipynb)
[![HuggingFace](https://img.shields.io/badge/%F0%9F%A4%97-Spaces-yellow)](https://huggingface.co/spaces/asterisk-labs/betaearth)

This notebook is the **flexible, multi-temporal** pipeline: pick any bounding box on an interactive map, download Sentinel-2 + Sentinel-1 + COP-DEM scenes from [Planetary Computer](https://planetarycomputer.microsoft.com), run per-timestamp inference, average into an annual 64-band GeoTIFF. Same pipeline the hosted Streamlit app uses.

**Other ways to try BetaEarth:**
- ⚡ **Fast mono-temporal quickstart**: [`demo.ipynb`](https://colab.research.google.com/github/asterisk-labs/beta-earth/blob/main/examples/demo.ipynb) — one Major TOM tile, no STAC downloads, runs in seconds.
- 🖥️  **Hosted app (no install)**: [huggingface.co/spaces/asterisk-labs/betaearth](https://huggingface.co/spaces/asterisk-labs/betaearth) — this notebook's pipeline wrapped in a Streamlit UI.
- ⚙️  **Local Streamlit app**: `streamlit run demo/app.py` from the [GitHub repo](https://github.com/asterisk-labs/beta-earth/tree/main/demo).

**What you'll get:**
- Annual average embedding (64-band GeoTIFF)
- Per-timestamp embeddings for each S2 / S1 scene
- PCA-RGB preview images for quick visual inspection

In [ ]:
# Runtime check — BetaEarth needs a GPU for fast inference (works on CPU but ~20x slower).
# This cell sets the notebook's preferred runtime to T4; Colab usually opens with GPU enabled
# on free tier, but if you see 'NO GPU' below, switch: Runtime -> Change runtime type -> T4 GPU.
import subprocess, sys
try:
    out = subprocess.check_output(["nvidia-smi", "-L"], stderr=subprocess.DEVNULL).decode()
    print(f"\u2713 GPU detected: {out.strip()}")
except Exception:
    print("\u26A0 NO GPU DETECTED. BetaEarth will run on CPU (~20x slower).")
    print("  Fix: Runtime \u2192 Change runtime type \u2192 T4 GPU \u2192 Save, then re-run.")

## 1. Install dependencies

In [ ]:
!pip install -qU "betaearth[generate]" ipyleaflet


## 2. Select area of interest

Draw a rectangle on the map below. Your bounding box will be captured automatically.

**Tip:** Keep the area small for fast results (< 30 x 30 km). Larger areas work but take longer to download and process.

In [ ]:
# Same initial view as the hosted Streamlit app: world zoom 2, centre (20, 0),
# Esri Imagery basemap with a CartoDB Light alternative. Draw any rectangle.
from ipyleaflet import Map, DrawControl, TileLayer, LayersControl
import ipywidgets as widgets
import numpy as np

esri = TileLayer(
    url="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attribution="Esri", name="Satellite", base=True,
)
carto_light = TileLayer(
    url="https://{s}.basemaps.cartocdn.com/light_all/{z}/{x}/{y}{r}.png",
    attribution="CartoDB", name="Light", base=True,
)

m = Map(
    center=(20.0, 0.0),   # world view — same as the hosted app
    zoom=2,
    layers=(esri, carto_light),
    scroll_wheel_zoom=True,
    layout=widgets.Layout(height="500px"),
)
m.add(LayersControl(position="topright"))

bbox_store = {}
draw = DrawControl(
    rectangle={"shapeOptions": {"color": "#ff6600", "weight": 2, "fillOpacity": 0.1}},
    polygon={}, polyline={}, circle={}, circlemarker={},
    edit=False, remove=False,
)

def on_draw(self, action, geo_json):
    if action != "created":
        return
    coords = geo_json["geometry"]["coordinates"][0]
    lons = [c[0] for c in coords]
    lats = [c[1] for c in coords]
    bbox_store["bbox"] = (min(lons), min(lats), max(lons), max(lats))
    w, s, e, n = bbox_store["bbox"]
    km_w = (e - w) * 111 * abs(np.cos(np.radians((s + n) / 2)))
    km_h = (n - s) * 111
    print(f"Selected bbox: W={w:.4f}, S={s:.4f}, E={e:.4f}, N={n:.4f}")
    print(f"  Size: {km_w:.1f} x {km_h:.1f} km")

draw.on_draw(on_draw)
m.add(draw)
m

If the interactive map doesn't render (e.g. on GitHub), set the bbox manually:

In [ ]:
# Uncomment and edit to set bbox manually (west, south, east, north):
# bbox_store["bbox"] = (13.18, 48.86, 13.65, 49.13)  # Bavarian Forest

if "bbox" not in bbox_store:
    print("Draw a rectangle on the map above, or set bbox_store['bbox'] manually.")
else:
    print(f"Using bbox: {bbox_store['bbox']}")

## 3. Configure and run

In [ ]:
# --- Settings ---
# Date range: 3 months by default. Short ranges are much faster and still
# give enough scenes for the multi-temporal averaging to saturate (~4 S2).
# Swap to a full year ("2023-01-01" -> "2023-12-31") if you want the
# seasonal-balanced annual aggregation like the Streamlit app defaults to.
START_DATE = "2023-06-01"
END_DATE   = "2023-08-31"

MAX_CLOUD    = 20    # % — hard cap on S2 cloud cover (eo:cloud_cover)
MIN_COVERAGE = 50    # % — only keep scenes covering at least this fraction of the AOI
N_S2         = 4     # max S2 scenes to use (picks lowest-cloud first)
N_S1         = 4     # max S1 scenes to use (date-ordered)
OUTPUT_DIR   = "outputs"
DEVICE       = "cuda"   # "cpu" if no GPU

assert "bbox" in bbox_store, "Set a bbox first (draw on map or set manually)"
bbox = bbox_store["bbox"]
print(f"Bbox:        {bbox}")
print(f"Date range:  {START_DATE} -> {END_DATE}")
print(f"Cloud cap:   {MAX_CLOUD}%")
print(f"Min cov:     {MIN_COVERAGE}%")

In [ ]:
# Turn on INFO logging from betaearth.generate so per-scene status prints inline.
import logging, time
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s", force=True)
logging.getLogger("betaearth.generate").setLevel(logging.INFO)

from pathlib import Path
import numpy as np
import pystac_client, planetary_computer

from betaearth import BetaEarth
from betaearth.generate import (
    compute_grid, download_s2, download_s1, download_dem,
    download_s2_cloud_mask, check_coverage,
)

# Load model + grid
model = BetaEarth.from_pretrained(device=DEVICE)
print(model)
grid = compute_grid(bbox)
h, w = grid["shape"]
print(f"Grid: {w} x {h} px  ({w*10/1000:.1f} x {h*10/1000:.1f} km)")

# STAC search over the chosen date range
PC_CATALOG = "https://planetarycomputer.microsoft.com/api/stac/v1"
catalog = pystac_client.Client.open(PC_CATALOG, modifier=planetary_computer.sign_inplace)
def _search(collection):
    return list(catalog.search(
        collections=[collection], bbox=list(bbox),
        datetime=f"{START_DATE}/{END_DATE}", max_items=200,
    ).items())

s2_items = _search("sentinel-2-l2a")
s1_items = _search("sentinel-1-rtc")
print(f"Found: {len(s2_items)} S2 + {len(s1_items)} S1 scenes in [{START_DATE}, {END_DATE}]")

# Keep up to N_S2 low-cloud S2 items + up to N_S1 S1 items
s2_items.sort(key=lambda x: x.properties.get("eo:cloud_cover", 100))
s2_items = [x for x in s2_items if x.properties.get("eo:cloud_cover", 100) < MAX_CLOUD][:N_S2]
s1_items.sort(key=lambda x: x.datetime)
s1_items = s1_items[:N_S1]
print(f"Kept: {len(s2_items)} S2 + {len(s1_items)} S1 scenes for inference")

# DEM (one static download for the AOI)
print("Downloading DEM ...")
dem = download_dem(grid)

# Per-scene inference — keep inputs + predictions for the visualisation step
scene_results = []   # list of dicts: date, sensor, s2_rgb, embedding, cos_with_annual (filled later)
emb_sum   = np.zeros((h, w, 64), dtype=np.float64)
emb_count = np.zeros((h, w), dtype=np.int32)

def _predict_and_accumulate(data_kw, doy, rgb):
    emb = model.predict(dem=dem, doy=doy, **data_kw)
    valid = np.linalg.norm(emb, axis=-1) > 1e-6
    mask = data_kw.pop("_cloud_mask", None)
    if mask is not None:
        valid &= mask
    emb_sum[valid] += emb[valid]
    emb_count[valid] += 1
    return emb, rgb

for item in s2_items:
    dt = item.datetime; doy = dt.timetuple().tm_yday
    cc = item.properties.get("eo:cloud_cover", 100)
    print(f"S2 {dt.date()}  cloud={cc:.0f}% ...")
    try:
        t0 = time.time(); s2 = download_s2(item, grid)
        cov = check_coverage(s2)
        if cov < MIN_COVERAGE:
            print(f"  skipped: {cov:.0f}% coverage"); continue
        mask = download_s2_cloud_mask(item, grid)
        emb = model.predict(s2_l2a=s2, dem=dem, doy=doy)
        valid = np.linalg.norm(emb, axis=-1) > 1e-6
        if mask is not None: valid &= mask
        emb_sum[valid] += emb[valid]; emb_count[valid] += 1
        # Store S2 RGB (B04, B03, B02) for the preview
        scene_results.append({
            "date": dt.date(), "sensor": "S2", "cloud": cc,
            "s2_rgb": s2[[2,1,0]],   # (3, H, W)
            "emb": emb,
        })
        print(f"  done  ({time.time()-t0:.1f}s)")
    except Exception as e:
        print(f"  FAILED: {type(e).__name__}: {e}")

for item in s1_items:
    dt = item.datetime; doy = dt.timetuple().tm_yday
    print(f"S1 {dt.date()} ...")
    try:
        t0 = time.time(); s1 = download_s1(item, grid)
        cov = check_coverage(s1)
        if cov < MIN_COVERAGE:
            print(f"  skipped: {cov:.0f}% coverage"); continue
        emb = model.predict(s1=s1, dem=dem, doy=doy)
        valid = np.linalg.norm(emb, axis=-1) > 1e-6
        emb_sum[valid] += emb[valid]; emb_count[valid] += 1
        scene_results.append({
            "date": dt.date(), "sensor": "S1", "cloud": None,
            "s2_rgb": None, "s1": s1, "emb": emb,
        })
        print(f"  done  ({time.time()-t0:.1f}s)")
    except Exception as e:
        print(f"  FAILED: {type(e).__name__}: {e}")

# Annual (really: date-range) average + L2 renorm
if emb_count.sum() == 0:
    raise RuntimeError("No scenes contributed to the aggregate — lower MIN_COVERAGE or widen date range.")
annual = emb_sum / np.maximum(emb_count[..., None], 1)
annual = (annual / np.clip(np.linalg.norm(annual, axis=-1, keepdims=True), 1e-8, None)).astype(np.float32)
print(f"\nAggregated {len(scene_results)} scenes -> annual embedding, max per-pixel count = {emb_count.max()}")

## 4. Visualise results

Display the annual PCA-RGB preview and per-timestamp previews side by side.

In [ ]:
# Visualise: S2 RGB (or S1 SAR-RGB) + BetaEarth PCA-RGB per scene,
# plus the aggregated annual PCA-RGB at the top.
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

def to_rgb_display(rgb_u16):
    """Stretch a uint16 (3, H, W) S2 RGB to displayable 0-1."""
    x = rgb_u16.astype(np.float32).transpose(1, 2, 0) / 3000.0
    return np.clip(x, 0, 1)

def sar_rgb(s1):
    """Build a false-colour RGB from Sentinel-1 linear power:
    R = VV (dB-stretched), G = VH, B = VV/VH ratio."""
    vv, vh = s1[0].astype(np.float32), s1[1].astype(np.float32)
    eps = 1e-6
    def _norm(a, p_lo=2, p_hi=98):
        valid = a > 0
        a = np.where(valid, 10 * np.log10(np.clip(a, eps, None)), np.nan)  # -> dB
        lo, hi = np.nanpercentile(a, [p_lo, p_hi])
        return np.clip((a - lo) / max(hi - lo, 1e-6), 0, 1)
    ratio = _norm(np.clip(vv, eps, None) / np.clip(vh, eps, None))
    return np.stack([_norm(vv), _norm(vh), ratio], axis=-1)

def pca_rgb(emb, pca=None):
    H, W, D = emb.shape
    flat = emb.reshape(-1, D)
    if pca is None:
        pca = PCA(n_components=3).fit(flat)
    rgb = pca.transform(flat).reshape(H, W, 3)
    for c in range(3):
        lo, hi = np.percentile(rgb[:, :, c], [2, 98])
        rgb[:, :, c] = np.clip((rgb[:, :, c] - lo) / max(hi - lo, 1e-8), 0, 1)
    return rgb, pca

# Fit PCA once on the annual embedding and reuse it for every scene
# so colours are directly comparable across timestamps and sensors.
annual_rgb, pca = pca_rgb(annual)

n = len(scene_results)
cols = min(max(n, 1), 5)
rows = 2   # top: input RGB (S2 optical / S1 SAR-RGB)  |  bottom: PCA-RGB
fig, axes = plt.subplots(rows + 1, cols, figsize=(cols * 3.2, (rows + 1) * 3.0))
if cols == 1:
    axes = np.array(axes).reshape(rows + 1, 1)

# Top-left: annual PCA-RGB spanning a single cell; hide the rest of row 0.
axes[0, 0].imshow(annual_rgb)
axes[0, 0].set_title(f"Annual PCA-RGB\n(avg of {n} scenes)", fontweight="bold")
axes[0, 0].axis("off")
for k in range(1, cols):
    axes[0, k].axis("off")

for i, s in enumerate(scene_results[:cols]):
    ax_rgb, ax_pca = axes[1, i], axes[2, i]
    if s["sensor"] == "S2" and s["s2_rgb"] is not None:
        ax_rgb.imshow(to_rgb_display(s["s2_rgb"]))
        ax_rgb.set_title(f"S2 {s['date']}  (cloud={s['cloud']:.0f}%)", fontsize=9)
    elif s["sensor"] == "S1" and s.get("s1") is not None:
        ax_rgb.imshow(sar_rgb(s["s1"]))
        ax_rgb.set_title(f"S1 {s['date']}  (VV/VH/ratio)", fontsize=9)
    else:
        ax_rgb.text(0.5, 0.5, f"{s['sensor']} {s['date']}", ha="center", va="center")
        ax_rgb.set_xlim(0, 1); ax_rgb.set_ylim(0, 1)
    ax_rgb.axis("off")
    scene_rgb, _ = pca_rgb(s["emb"], pca)
    ax_pca.imshow(scene_rgb)
    ax_pca.set_title("BetaEarth PCA-RGB", fontsize=9)
    ax_pca.axis("off")

for j in range(len(scene_results), cols):
    axes[1, j].axis("off"); axes[2, j].axis("off")

plt.suptitle(f"BetaEarth — {START_DATE} to {END_DATE}", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

## 5. Save outputs + download ZIP

Writes the annual embedding (64-band GeoTIFF), the per-scene embeddings, all preview PNGs, and a manifest into `outputs/<bbox>/`, then zips and offers a direct browser download (Colab only).

In [ ]:
import json, zipfile
from pathlib import Path
from PIL import Image
import betaearth
from betaearth.generate import write_geotiff

OUT = Path(f'outputs/bbox_{bbox[0]:.4f}_{bbox[1]:.4f}')
OUT.mkdir(parents=True, exist_ok=True)

# 1. Annual 64-band embedding as Cloud-Optimised GeoTIFF (CRS + transform from grid)
write_geotiff(annual, grid, OUT / 'annual.tif', band_first=False)
Image.fromarray((annual_rgb * 255).astype(np.uint8)).save(OUT / 'annual_preview_pca.png')

# 2. Per-scene embeddings + previews
scenes_dir = OUT / 'scenes'; scenes_dir.mkdir(exist_ok=True)
scene_entries = []
for s in scene_results:
    tag = f"{s['date']}_{s['sensor']}"
    sdir = scenes_dir / tag; sdir.mkdir(exist_ok=True)
    write_geotiff(s['emb'].astype(np.float32), grid, sdir / 'embedding.tif', band_first=False)
    if s['sensor'] == 'S2' and s.get('s2_rgb') is not None:
        Image.fromarray((to_rgb_display(s['s2_rgb']) * 255).astype(np.uint8)).save(sdir / 'preview_rgb.png')
    elif s['sensor'] == 'S1' and s.get('s1') is not None:
        Image.fromarray((sar_rgb(s['s1']) * 255).astype(np.uint8)).save(sdir / 'preview_sar_rgb.png')
    scene_rgb, _ = pca_rgb(s['emb'], pca)
    Image.fromarray((scene_rgb * 255).astype(np.uint8)).save(sdir / 'preview_pca.png')
    scene_entries.append({'date': str(s['date']), 'sensor': s['sensor'],
                          'cloud': s.get('cloud'), 'embedding_file': f'scenes/{tag}/embedding.tif'})

# 3. Manifest
meta = {
    'bbox_4326':     list(bbox),
    'date_range':    [START_DATE, END_DATE],
    'grid': {'shape': list(grid['shape']), 'crs': str(grid['crs'])},
    'n_scenes':      len(scene_entries),
    'scenes':        scene_entries,
    'model_hf_repo': 'asterisk-labs/betaearth-segformer-film-robust',
    'betaearth':     betaearth.__version__,
    'aggregation':   'per-timestamp predict -> pixel-wise mean (cloud-masked) -> L2 renorm',
}
with open(OUT / 'manifest.json', 'w') as f:
    json.dump(meta, f, indent=2)

# 4. Zip the whole AOI directory
zip_path = Path(f'outputs/betaearth_{bbox[0]:.4f}_{bbox[1]:.4f}.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(OUT.rglob('*')):
        if p.is_file():
            zf.write(p, p.relative_to(OUT.parent))
size_mb = zip_path.stat().st_size / (1024 * 1024)
print(f'Wrote {zip_path}  ({size_mb:.1f} MB, {len(scene_entries)} scenes)')

# 5. Trigger browser download in Colab
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    print(f'(not in Colab — the ZIP is at ./{zip_path} in the file browser)')

## Next steps

**Simpler starting point:**
- ⚡ [`demo.ipynb`](https://colab.research.google.com/github/asterisk-labs/beta-earth/blob/main/examples/demo.ipynb) — single Major TOM tile, one predict call, no STAC downloads. Good if you want to understand the model without the acquisition pipeline.

**Scale up:**
- **Save outputs** — the ZIP download (cell above) contains `annual.tif` (georeferenced 64-band COG), per-scene `embedding.tif`s, previews, and a `manifest.json`. For larger batches, `outputs/` stays on the Colab runtime's disk and is browsable in the file pane.
- **Open in QGIS** — the 64-band `embedding.tif` files are georeferenced COGs.
- **Use from CLI** for larger areas or batch processing:
  ```bash
  pip install betaearth[generate]
  betaearth-generate --bbox 13.18 48.86 13.65 49.13 --years 2023
  ```
- 🖥️  **Hosted Streamlit app** — try [huggingface.co/spaces/asterisk-labs/betaearth](https://huggingface.co/spaces/asterisk-labs/betaearth) for browser-based generation; or run the same app locally with `streamlit run demo/app.py`.